# Iowa Corn Monthly Yield Model Improvement

This Colab notebook improves the monthly-grain Iowa Corn yield model by adding leakage-safe historical yield features. It does not rerun extraction and does not touch the batch pipeline.

Goal: train on 2017-2021, test on 2022, and check whether the improved ML model beats `BaselinePreviousYearSameCounty`.

In [ ]:
!pip -q install pandas numpy scikit-learn joblib matplotlib pyarrow

import json
import math
import os
import re
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

print('Python:', sys.version)
print('pandas:', pd.__version__)
import sklearn
print('scikit-learn:', sklearn.__version__)

## Paths

Set `USE_DRIVE = True` if your files are in Google Drive. Otherwise, the notebook will prompt you to upload the merged parquet file and USDA CSVs.

In [ ]:
USE_DRIVE = True
DRIVE_ROOT = Path('/content/drive/MyDrive/cropnet_iowa_corn')
REPO_ROOT = DRIVE_ROOT / 'Crop-Net-repo'

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print('Drive mount skipped or unavailable:', exc)

MONTHLY_PATH = REPO_ROOT / 'outputs/experiments/corn_ia_monthly_2017_2022/artifacts/official_monthly_feature_table.parquet'
OUTPUT_DIR = REPO_ROOT / 'outputs/yield_baseline/corn_ia_2017_2022_monthly_improved'
PLOTS_DIR = OUTPUT_DIR / 'plots'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

USDA_PATHS = [
    REPO_ROOT / f'data/usda_labels/USDA Crop Dataset/Corn/{year}/USDA_Corn_County_{year}.csv'
    for year in range(2017, 2023)
]

print('MONTHLY_PATH:', MONTHLY_PATH)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('USDA files found:', sum(p.exists() for p in USDA_PATHS), '/', len(USDA_PATHS))

## Load Monthly Features

If the Drive path is missing, upload `official_monthly_feature_table.parquet` when prompted.

In [ ]:
if not MONTHLY_PATH.exists():
    print('Drive parquet not found. Upload official_monthly_feature_table.parquet now.')
    from google.colab import files
    uploaded = files.upload()
    parquet_files = [name for name in uploaded if name.lower().endswith('.parquet')]
    if not parquet_files:
        raise FileNotFoundError('No parquet file uploaded.')
    MONTHLY_PATH = Path(parquet_files[0])

df = pd.read_parquet(MONTHLY_PATH)
print('Loaded monthly table:', df.shape)
display(df.head())
print('Columns:', list(df.columns))

## Label Handling

The monthly feature table may not contain yield labels. If no target is found, upload the six USDA Corn county CSV files for 2017-2022 or keep them in the Drive path listed above.

In [ ]:
YIELD_COLUMN_CANDIDATES = ['yield_bu_acre', 'YIELD, MEASURED IN BU / ACRE', 'yield', 'target_value']

def normalize_county_id(values):
    s = pd.Series(values).astype(str).str.extract(r'(\d+)', expand=False).fillna('')
    return s.str.zfill(5)

def normalize_crop_type(values):
    return pd.Series(values).astype(str).str.strip().str.lower().str.replace('_', ' ', regex=False).str.replace('-', ' ', regex=False)

def detect_yield_column(columns):
    for candidate in YIELD_COLUMN_CANDIDATES:
        if candidate in columns:
            return candidate
    for col in columns:
        lowered = str(col).lower()
        if 'yield' in lowered and ('acre' in lowered or lowered == 'yield'):
            return col
    return None

def infer_crop_from_filename(name):
    match = re.search(r'USDA_([^_]+(?:_[^_]+)*)_County_\d{4}\.csv$', Path(name).name)
    if match:
        return match.group(1).replace('_', ' ').lower()
    return 'corn'

def load_usda_label_csv(path):
    raw = pd.read_csv(path, dtype=str)
    yield_col = detect_yield_column(raw.columns)
    if yield_col is None:
        raise ValueError(f'Could not detect yield column in {path}. Columns: {list(raw.columns)}')
    lower_cols = {c.lower(): c for c in raw.columns}
    if 'year' in lower_cols:
        year = pd.to_numeric(raw[lower_cols['year']], errors='coerce')
    else:
        year_match = re.search(r'(20\d{2}|19\d{2})', Path(path).name)
        inferred_year = int(year_match.group(1)) if year_match else np.nan
        year = pd.Series([inferred_year] * len(raw))
    if 'state_ansi' in lower_cols and 'county_ansi' in lower_cols:
        state = raw[lower_cols['state_ansi']].astype(str).str.extract(r'(\d+)', expand=False).str.zfill(2)
        county = raw[lower_cols['county_ansi']].astype(str).str.extract(r'(\d+)', expand=False).str.zfill(3)
        county_id = state + county
    elif 'county_id' in lower_cols:
        county_id = normalize_county_id(raw[lower_cols['county_id']])
    elif 'fips' in lower_cols:
        county_id = normalize_county_id(raw[lower_cols['fips']])
    else:
        raise ValueError(f'Could not detect county FIPS columns in {path}.')
    crop_type = raw[lower_cols['commodity_desc']].str.lower() if 'commodity_desc' in lower_cols else infer_crop_from_filename(path)
    labels = pd.DataFrame({
        'county_id': county_id.astype(str).str.zfill(5),
        'year': pd.to_numeric(year, errors='coerce').astype('Int64'),
        'crop_type': normalize_crop_type(crop_type),
        'yield_bu_acre': pd.to_numeric(raw[yield_col], errors='coerce'),
        'target_unit': 'BU / ACRE',
    })
    return labels.dropna(subset=['county_id', 'year', 'yield_bu_acre'])

target_col = detect_yield_column(df.columns)
print('Detected target column:', target_col)

df['county_id'] = normalize_county_id(df['county_id']) if 'county_id' in df.columns else normalize_county_id(df.get('fips', ''))
df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
df['month'] = pd.to_numeric(df['month'], errors='coerce').astype('Int64')
df['crop_type'] = normalize_crop_type(df['crop_type']) if 'crop_type' in df.columns else 'corn'

if target_col is None:
    existing_usda = [p for p in USDA_PATHS if p.exists()]
    if len(existing_usda) < 6:
        print('Upload USDA_Corn_County_2017.csv through USDA_Corn_County_2022.csv now.')
        from google.colab import files
        uploaded = files.upload()
        existing_usda = [Path(name) for name in uploaded if name.lower().endswith('.csv')]
    usda = pd.concat([load_usda_label_csv(path) for path in existing_usda], ignore_index=True)
    usda = usda[usda['crop_type'].eq('corn')].copy()
    df = df.merge(usda, on=['county_id', 'year', 'crop_type'], how='inner')
    target_col = 'yield_bu_acre'
else:
    if target_col != 'yield_bu_acre':
        df['yield_bu_acre'] = pd.to_numeric(df[target_col], errors='coerce')
        target_col = 'yield_bu_acre'
    else:
        df[target_col] = pd.to_numeric(df[target_col], errors='coerce')

df = df.dropna(subset=[target_col, 'county_id', 'year', 'month']).copy()
df['year'] = df['year'].astype(int)
df['month'] = df['month'].astype(int)
print('Training frame after label handling:', df.shape)
print('Years:', sorted(df['year'].unique()))
print('Months:', sorted(df['month'].unique()))
print('Counties:', df['county_id'].nunique())
display(df[target_col].describe())

## Validation Before Modeling

In [ ]:
FORBIDDEN_COLUMNS = {'forecast_step', 'known_months', 'source_note', 'y_pred'}
forbidden_present = sorted(FORBIDDEN_COLUMNS & set(df.columns))
duplicate_count = df.duplicated(['county_id', 'crop_type', 'year', 'month']).sum()

print('Forbidden forecast columns:', forbidden_present)
print('Duplicate county/crop/year/month rows:', duplicate_count)
print('Rows:', len(df))
print('Numeric NaN count:', int(df.select_dtypes(include=[np.number]).isna().sum().sum()))
print('Numeric inf count:', int(np.isinf(df.select_dtypes(include=[np.number]).to_numpy()).sum()))

assert not forbidden_present, f'Forecast-generated columns found: {forbidden_present}'
assert duplicate_count == 0, 'Duplicate monthly keys found.'
assert 2022 in set(df['year']), 'Test year 2022 is missing.'

## Leakage-Safe Historical Yield Features

In [ ]:
def build_historical_yield_features(frame, target_col='yield_bu_acre'):
    annual = (
        frame[['county_id', 'crop_type', 'year', target_col]]
        .drop_duplicates(['county_id', 'crop_type', 'year'])
        .sort_values(['county_id', 'crop_type', 'year'])
        .copy()
    )

    enriched = frame.copy()
    for lag in [1, 2, 3]:
        lag_table = annual[['county_id', 'crop_type', 'year', target_col]].copy()
        lag_table['year'] = lag_table['year'] + lag
        lag_table = lag_table.rename(columns={target_col: f'yield_lag_{lag}_same_county'})
        lag_table[f'yield_lag_{lag}_source_year'] = lag_table['year'] - lag
        enriched = enriched.merge(lag_table, on=['county_id', 'crop_type', 'year'], how='left')

    stats = annual.copy()
    group = stats.groupby(['county_id', 'crop_type'], group_keys=False)
    prior = group[target_col].shift(1)
    stats['yield_roll_mean_3yr_same_county'] = prior.groupby([stats['county_id'], stats['crop_type']]).rolling(3, min_periods=1).mean().reset_index(level=[0, 1], drop=True)
    stats['yield_roll_std_3yr_same_county'] = prior.groupby([stats['county_id'], stats['crop_type']]).rolling(3, min_periods=2).std().reset_index(level=[0, 1], drop=True)

    def slope(values):
        arr = pd.Series(values).dropna().to_numpy(dtype=float)
        if len(arr) < 2:
            return np.nan
        return float(np.polyfit(np.arange(len(arr)), arr, 1)[0])

    stats['yield_trend_3yr_same_county'] = prior.groupby([stats['county_id'], stats['crop_type']]).rolling(3, min_periods=2).apply(slope, raw=False).reset_index(level=[0, 1], drop=True)
    stats = stats[['county_id', 'crop_type', 'year', 'yield_roll_mean_3yr_same_county', 'yield_roll_std_3yr_same_county', 'yield_trend_3yr_same_county']]
    enriched = enriched.merge(stats, on=['county_id', 'crop_type', 'year'], how='left')

    state_prev = annual.groupby(['crop_type', 'year'], as_index=False)[target_col].mean()
    state_prev['year'] = state_prev['year'] + 1
    state_prev = state_prev.rename(columns={target_col: 'state_mean_yield_prev_year'})
    enriched = enriched.merge(state_prev, on=['crop_type', 'year'], how='left')
    enriched['county_vs_state_yield_prev_year'] = enriched['yield_lag_1_same_county'] - enriched['state_mean_yield_prev_year']
    return enriched

df = build_historical_yield_features(df, target_col=target_col)
for lag in [1, 2, 3]:
    source_col = f'yield_lag_{lag}_source_year'
    valid = df[source_col].notna()
    assert (df.loc[valid, source_col] < df.loc[valid, 'year']).all(), f'Leakage in {source_col}'

df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12.0)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12.0)

history_cols = [
    'yield_lag_1_same_county',
    'yield_lag_2_same_county',
    'yield_lag_3_same_county',
    'yield_roll_mean_3yr_same_county',
    'yield_roll_std_3yr_same_county',
    'yield_trend_3yr_same_county',
    'state_mean_yield_prev_year',
    'county_vs_state_yield_prev_year',
]
print('Historical feature missing rates:')
display(df[history_cols].isna().mean().sort_values(ascending=False))
display(df[['county_id', 'year', 'month', target_col] + history_cols].head(15))

## Feature Groups And Time Split

In [ ]:
TRAIN_YEARS = [2017, 2018, 2019, 2020, 2021]
TEST_YEAR = 2022

metadata_tokens = ['county', 'name', 'fips', 'ansi', 'state', 'crop', 'unit', 'target', 'yield', 'source_year']
identifier_cols = {'year', 'month', 'county_id', 'crop_type', target_col, 'target_unit'}

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
remote_cols = []
for col in numeric_cols:
    lower = col.lower()
    if col in identifier_cols or col in history_cols or lower.endswith('_source_year'):
        continue
    if any(token in lower for token in metadata_tokens):
        continue
    remote_cols.append(col)

for col in ['month', 'month_sin', 'month_cos']:
    if col not in remote_cols and col in df.columns:
        remote_cols.append(col)

feature_groups = {
    'remote_only': remote_cols,
    'history_only': history_cols,
    'combined': remote_cols + history_cols,
}

train_df = df[df['year'].isin(TRAIN_YEARS)].copy()
test_df = df[df['year'].eq(TEST_YEAR)].copy()

print('Train years:', sorted(train_df['year'].unique()))
print('Test years:', sorted(test_df['year'].unique()))
print('Train rows:', len(train_df), 'Test rows:', len(test_df))
print('Train counties:', train_df['county_id'].nunique(), 'Test counties:', test_df['county_id'].nunique())
for name, cols in feature_groups.items():
    print(name, len(cols), 'features')

assert sorted(train_df['year'].unique()) == TRAIN_YEARS
assert sorted(test_df['year'].unique()) == [TEST_YEAR]
assert len(test_df) > 0
assert len(feature_groups['combined']) > 0

## Train And Evaluate Improved Models

In [ ]:
def mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = y_true != 0
    if not mask.any():
        return np.nan
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100.0)

def regression_metrics(y_true, y_pred):
    return {
        'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'r2': float(r2_score(y_true, y_pred)),
        'mape': mape(y_true, y_pred),
    }

def make_models():
    return {
        'Ridge': Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('model', Ridge(alpha=10.0)),
        ]),
        'RandomForest': Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('model', RandomForestRegressor(n_estimators=500, min_samples_leaf=2, random_state=42, n_jobs=-1)),
        ]),
        'ExtraTrees': Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('model', ExtraTreesRegressor(n_estimators=500, min_samples_leaf=2, random_state=42, n_jobs=-1)),
        ]),
    }

y_train = train_df[target_col].astype(float)
y_test = test_df[target_col].astype(float)

results = []
prediction_frame = test_df[['county_id', 'crop_type', 'year', 'month', target_col]].copy()
prediction_frame = prediction_frame.rename(columns={target_col: 'actual'})
if 'county_name' in test_df.columns:
    prediction_frame['county_name'] = test_df['county_name']
elif 'county' in test_df.columns:
    prediction_frame['county_name'] = test_df['county']

train_mean_pred = np.full(len(test_df), float(y_train.mean()))
prev_year_pred = test_df['yield_lag_1_same_county'].fillna(y_train.mean()).to_numpy(dtype=float)
prediction_frame['BaselineTrainMean'] = train_mean_pred
prediction_frame['BaselinePreviousYearSameCounty'] = prev_year_pred

for baseline_name, pred in [('BaselineTrainMean', train_mean_pred), ('BaselinePreviousYearSameCounty', prev_year_pred)]:
    row = {'model': baseline_name, 'model_type': 'baseline', 'feature_group': 'baseline', 'train_rows': len(train_df), 'test_rows': len(test_df)}
    row.update(regression_metrics(y_test, pred))
    results.append(row)

fitted_models = {}
for group_name, cols in feature_groups.items():
    X_train = train_df[cols].replace([np.inf, -np.inf], np.nan)
    X_test = test_df[cols].replace([np.inf, -np.inf], np.nan)
    for model_name, model in make_models().items():
        run_name = f'{group_name}_{model_name}'
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        prediction_frame[run_name] = pred
        fitted_models[run_name] = {'model': model, 'features': cols, 'feature_group': group_name, 'model_name': model_name}
        row = {'model': run_name, 'model_type': 'ml', 'feature_group': group_name, 'train_rows': len(train_df), 'test_rows': len(test_df)}
        row.update(regression_metrics(y_test, pred))
        results.append(row)

metrics = pd.DataFrame(results).sort_values(['rmse', 'mae']).reset_index(drop=True)
display(metrics)

ml_metrics = metrics[metrics['model_type'].eq('ml')].copy()
best_row = ml_metrics.sort_values('rmse').iloc[0]
best_model_name = best_row['model']
best_bundle = fitted_models[best_model_name]
prediction_frame['best_prediction'] = prediction_frame[best_model_name]
prediction_frame['best_residual'] = prediction_frame['best_prediction'] - prediction_frame['actual']

baseline_row = metrics[metrics['model'].eq('BaselinePreviousYearSameCounty')].iloc[0]
best_combined = metrics[(metrics['model_type'].eq('ml')) & (metrics['feature_group'].eq('combined'))].sort_values('rmse').iloc[0]
beats_previous_year = bool(
    (best_combined['rmse'] < baseline_row['rmse']) and
    (best_combined['mae'] < baseline_row['mae']) and
    (best_combined['mape'] < baseline_row['mape'])
)

print('Best ML model:', best_model_name)
print('Best combined model:', best_combined['model'])
print('Previous-year baseline RMSE/MAE/MAPE:', baseline_row[['rmse', 'mae', 'mape']].to_dict())
print('Best combined RMSE/MAE/MAPE:', best_combined[['rmse', 'mae', 'mape']].to_dict())
print('Combined ML beats previous-year baseline on RMSE, MAE, and MAPE:', beats_previous_year)

## Month And Window Benchmarks

In [ ]:
def benchmark_subset(frame, pred_col='best_prediction'):
    return regression_metrics(frame['actual'], frame[pred_col])

month_rows = []
for month, sub in prediction_frame.groupby('month'):
    row = {'month': int(month), 'rows': len(sub)}
    row.update(benchmark_subset(sub))
    month_rows.append(row)
month_benchmark = pd.DataFrame(month_rows).sort_values('month')
display(month_benchmark)

windows = {
    'Jan': [1],
    'Jan-Mar': [1, 2, 3],
    'Apr-Jun': [4, 5, 6],
    'Apr-Sep': [4, 5, 6, 7, 8, 9],
    'FullYear': list(range(1, 13)),
}
window_rows = []
group_cols = ['county_id', 'crop_type', 'year']
for window_name, months in windows.items():
    sub = prediction_frame[prediction_frame['month'].isin(months)].copy()
    agg = sub.groupby(group_cols, as_index=False).agg(actual=('actual', 'first'), best_prediction=('best_prediction', 'mean'), baseline_previous_year=('BaselinePreviousYearSameCounty', 'mean'))
    row = {'window': window_name, 'months': ','.join(map(str, months)), 'rows': len(sub), 'county_years': len(agg)}
    row.update({f'ml_{k}': v for k, v in regression_metrics(agg['actual'], agg['best_prediction']).items()})
    row.update({f'previous_year_{k}': v for k, v in regression_metrics(agg['actual'], agg['baseline_previous_year']).items()})
    window_rows.append(row)
window_benchmark = pd.DataFrame(window_rows)
display(window_benchmark)

## Save Improved Artifacts

In [ ]:
metrics.to_csv(OUTPUT_DIR / 'improved_metrics.csv', index=False)
prediction_frame.to_csv(OUTPUT_DIR / 'improved_predictions_2022.csv', index=False)
month_benchmark.to_csv(OUTPUT_DIR / 'month_benchmark.csv', index=False)
window_benchmark.to_csv(OUTPUT_DIR / 'window_benchmark.csv', index=False)

feature_group_comparison = (
    metrics[metrics['model_type'].eq('ml')]
    .sort_values('rmse')
    .groupby('feature_group', as_index=False)
    .first()
    .sort_values('rmse')
)
feature_group_comparison.to_csv(OUTPUT_DIR / 'feature_group_comparison.csv', index=False)

joblib.dump(best_bundle['model'], OUTPUT_DIR / 'best_improved_yield_model.joblib')
(OUTPUT_DIR / 'feature_columns.txt').write_text('\n'.join(best_bundle['features']))

metadata = {
    'target_grain': 'monthly',
    'label_strategy': 'annual_yield_copied_to_months',
    'uses_forecast_generated_features': False,
    'split_strategy': 'year_split',
    'train_years': TRAIN_YEARS,
    'test_year': TEST_YEAR,
    'train_rows': int(len(train_df)),
    'test_rows': int(len(test_df)),
    'county_count_train': int(train_df['county_id'].nunique()),
    'county_count_test': int(test_df['county_id'].nunique()),
    'best_model': best_model_name,
    'best_feature_group': best_bundle['feature_group'],
    'best_combined_model': str(best_combined['model']),
    'beats_previous_year_same_county': beats_previous_year,
    'history_features': history_cols,
    'feature_columns': best_bundle['features'],
}
(OUTPUT_DIR / 'improved_model_metadata.json').write_text(json.dumps(metadata, indent=2))

print('Saved improved artifacts to:', OUTPUT_DIR)
display(feature_group_comparison)

## Plots

In [ ]:
def save_bar(metric, lower_is_better=True):
    data = metrics.sort_values(metric, ascending=lower_is_better)
    plt.figure(figsize=(10, 5))
    plt.bar(data['model'], data[metric])
    plt.xticks(rotation=65, ha='right')
    direction = 'lower is better' if lower_is_better else 'higher is better'
    plt.title(f'Model comparison: {metric.upper()} ({direction})')
    plt.ylabel(metric.upper())
    plt.tight_layout()
    path = PLOTS_DIR / f'model_comparison_{metric}.png'
    plt.savefig(path, dpi=160)
    plt.show()
    print('Saved', path)

for metric, lower in [('rmse', True), ('mae', True), ('mape', True), ('r2', False)]:
    save_bar(metric, lower)

plt.figure(figsize=(6, 6))
plt.scatter(prediction_frame['actual'], prediction_frame['best_prediction'], alpha=0.65)
low = min(prediction_frame['actual'].min(), prediction_frame['best_prediction'].min())
high = max(prediction_frame['actual'].max(), prediction_frame['best_prediction'].max())
plt.plot([low, high], [low, high], 'k--', label='y=x')
plt.xlabel('Actual yield')
plt.ylabel('Predicted yield')
plt.title(f'Actual vs predicted: {best_model_name}')
plt.legend()
plt.tight_layout()
path = PLOTS_DIR / 'actual_vs_predicted.png'
plt.savefig(path, dpi=160)
plt.show()
print('Saved', path)

plt.figure(figsize=(8, 5))
plt.hist(prediction_frame['best_residual'], bins=30, edgecolor='black')
plt.axvline(0, color='red', linestyle='--')
plt.title('Residual distribution: predicted - actual')
plt.xlabel('Residual, positive means overprediction')
plt.ylabel('Rows')
plt.tight_layout()
path = PLOTS_DIR / 'residual_distribution.png'
plt.savefig(path, dpi=160)
plt.show()
print('Saved', path)

county_label = 'county_name' if 'county_name' in prediction_frame.columns else 'county_id'
county_errors = prediction_frame.groupby(['county_id'], as_index=False).agg(actual=('actual', 'first'), predicted=('best_prediction', 'mean'))
if county_label == 'county_name':
    names = prediction_frame[['county_id', 'county_name']].drop_duplicates('county_id')
    county_errors = county_errors.merge(names, on='county_id', how='left')
else:
    county_errors['county_name'] = county_errors['county_id']
county_errors['error'] = county_errors['predicted'] - county_errors['actual']
county_errors['abs_error'] = county_errors['error'].abs()
worst20 = county_errors.sort_values('abs_error', ascending=False).head(20)
display(worst20)
plt.figure(figsize=(10, 6))
plt.barh(worst20['county_name'].astype(str), worst20['abs_error'])
plt.gca().invert_yaxis()
plt.xlabel('Absolute error')
plt.title('Worst 20 counties by absolute error')
plt.tight_layout()
path = PLOTS_DIR / 'worst_county_errors.png'
plt.savefig(path, dpi=160)
plt.show()
print('Saved', path)

final_estimator = best_bundle['model'].named_steps.get('model', best_bundle['model']) if hasattr(best_bundle['model'], 'named_steps') else best_bundle['model']
if hasattr(final_estimator, 'feature_importances_'):
    importances = pd.DataFrame({'feature': best_bundle['features'], 'importance': final_estimator.feature_importances_})
    top20 = importances.sort_values('importance', ascending=False).head(20)
    display(top20)
    plt.figure(figsize=(10, 6))
    plt.barh(top20['feature'], top20['importance'])
    plt.gca().invert_yaxis()
    plt.xlabel('Importance')
    plt.title('Top 20 feature importances')
    plt.tight_layout()
    path = PLOTS_DIR / 'feature_importance_top20.png'
    plt.savefig(path, dpi=160)
    plt.show()
    print('Saved', path)
else:
    print(f'{best_model_name} does not expose tree-style feature_importances_.')

## Final Report

In [ ]:
print('Output directory:', OUTPUT_DIR)
print('Best ML model:', best_model_name)
print('Best ML feature group:', best_bundle['feature_group'])
print('Best combined model:', best_combined['model'])
print('Combined ML beats BaselinePreviousYearSameCounty:', beats_previous_year)
print('Uses forecast-generated features:', metadata['uses_forecast_generated_features'])
print('\nMetrics:')
display(metrics)
print('\nFeature group comparison:')
display(feature_group_comparison)
if not beats_previous_year:
    print('Interpretation: the previous-year same-county yield baseline remains stronger. Historical yield is likely the dominant signal, and monthly CropNet features are not yet adding enough incremental value.')
else:
    print('Interpretation: the combined ML model improved over the previous-year same-county baseline on RMSE, MAE, and MAPE.')